In [120]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import numpy as np

# Configuration
CFG = {
    "esm_dim": 1280,      # ESM-2 typical output
    "gvp_dim": 1084,       # Example GVP scalar output dim
    "egnn_dim": 256,      # EGNN node feature dim
    "chemberta_dim": 384, # ChemBERTa typical output
    "latent_dim": 256,    # Final shared latent space size
    "dropout": 0.1
}

In [121]:
class GatedFusion(nn.Module):
    def __init__(self, dim_1, dim_2, output_dim, dropout):
        super().__init__()
        combined_dim = dim_1 + dim_2
        # Gate: determines "how much" of each feature to let through
        self.gate = nn.Sequential(
            nn.Linear(combined_dim, output_dim),
            nn.Sigmoid()
        )
        # Transformation: the actual feature processing
        self.output_layer = nn.Sequential(
            nn.Linear(combined_dim, output_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

    def forward(self, x1, x2):
        combined = torch.cat([x1, x2], dim=-1)
        g = self.gate(combined)
        f = self.output_layer(combined)
        return g * f  # Element-wise gating 

In [122]:
class BioMultitaskModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.protein_encoder = GatedFusion(
            config["esm_dim"], config["gvp_dim"], config["latent_dim"], CFG['dropout']
        )
        self.drug_encoder = GatedFusion(
            config["chemberta_dim"], config["egnn_dim"], config["latent_dim"], CFG['dropout']
        )
        
        self.protein_projector = nn.Sequential(
            nn.Linear(config["latent_dim"], config["latent_dim"]),
            nn.BatchNorm1d(config["latent_dim"]),
            nn.ReLU(),
            nn.Linear(config["latent_dim"], config["latent_dim"])
        )
        
        self.drug_projector = nn.Sequential(
            nn.Linear(config["latent_dim"], config["latent_dim"]),
            nn.BatchNorm1d(config["latent_dim"]),
            nn.ReLU(),
            nn.Linear(config["latent_dim"], config["latent_dim"])
        )

    def forward(self, esm, gvp, chem, egnn):
        p_fused = self.protein_encoder(esm, gvp)
        z_p = self.protein_projector(p_fused)
        
        d_fused = self.drug_encoder(chem, egnn)
        z_d = self.drug_projector(d_fused)
        
        z_p = F.normalize(z_p, p=2, dim=-1)
        z_d = F.normalize(z_d, p=2, dim=-1)
        
        return z_p, z_d


In [ ]:
class MultiTaskLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature
        self.mse = nn.MSELoss()

    def dti_contrastive_loss(self, z_p, z_d):
        """
        Standard InfoNCE loss.
        z_p: Protein embeddings [batch, latent_dim]
        z_d: Drug embeddings [batch, latent_dim]
        """
        batch_size = z_p.shape[0]
        
        # Compute similarity matrix (Cosine similarity because vectors are normalized)
        logits = torch.matmul(z_d, z_p.T) / self.temperature
        
        # Ground truth is the diagonal (each drug matches its own protein in the batch)
        labels = torch.arange(batch_size).to(z_p.device)
        
        loss_p = F.cross_entropy(logits, labels)
        loss_d = F.cross_entropy(logits.T, labels)
        
        return (loss_p + loss_d) / 2

In [124]:
class ADRManager:
    def __init__(self, latent_dim):
        # Dictionary to store {adr_id: list_of_embeddings}
        self.adr_registry = {}

        # Final averaged prototypes {adr_id: tensor_point}
        self.prototypes = {}

    def update_registry(self, adr_ids, drug_embeddings, protein_embeddings):
        """
        Call this during an epoch to collect embeddings for each ADR.
        As per your plan: ADR = avg(Drugs + Proteins involved)
        """
        for i, adr_id in enumerate(adr_ids):
            if adr_id not in self.adr_registry:
                self.adr_registry[adr_id] = []
            
            # Combine the drug and the protein it interacted with for this ADR
            combined_context = (drug_embeddings[i] + protein_embeddings[i]) / 2
            self.adr_registry[adr_id].append(combined_context.detach())

    def compute_prototypes(self):
        """Compute the final 'point' for every ADR in the latent space"""
        for adr_id, embeddings in self.adr_registry.items():
            self.prototypes[adr_id] = torch.stack(embeddings).mean(dim=0)
            
        return self.prototypes

In [125]:

class ADRData:
    def __init__(self, id_to_name_dict):
        """
        Initializes with a dictionary of {meddra_id: meddra_name}.
        """
        self.id_to_name = id_to_name_dict
        self.unique_ids = sorted(list(id_to_name_dict.keys()))
        
        self.id_to_idx = {adr_id: i for i, adr_id in enumerate(self.unique_ids)}
        self.idx_to_id = {i: adr_id for i, adr_id in enumerate(self.unique_ids)}
        
        self.vocab_size = len(self.unique_ids)

    def encode(self, adr_list):
        """
        Takes a list of ADR IDs and returns a binary vector (1s and 0s).
        Example: ['10028553', '10003041'] -> [0, 1, 0, 0, 1...]
        """
        vector = np.zeros(self.vocab_size, dtype=np.int8)
        
        for adr_id in adr_list:
            if adr_id in self.id_to_idx:
                idx = self.id_to_idx[adr_id]
                vector[idx] = 1
            else:
                print(f"Warning: ADR ID {adr_id} not in vocabulary.")
                
        return vector

    def decode(self, vector):
        """
        Takes a binary vector and returns a list of human-readable ADR names.
        """
        decoded_names = []
        
        active_indices = np.where(vector == 1)[0]
        
        for idx in active_indices:
            adr_id = self.idx_to_id[idx]
            name = self.id_to_name.get(adr_id, "Unknown ADR")
            decoded_names.append(name)
            
        return decoded_names


In [126]:
class DTIDataset(Dataset):
    def __init__(self, parquet_path, adr_manager, adr_prototype_map=None):
        """
        Args:
            parquet_path: Path to your drug-protein-adr parquet.
            adr_manager: An instance of your ADRData class.
            adr_prototype_map: {index: tensor_embedding} mapping for prototyping.
        """
        self.data = pd.read_parquet(parquet_path)
        self.adr_manager = adr_manager
        self.adr_map = adr_prototype_map 

        if len(self.data) > 0:
            sample = self.data.iloc[0]
            # Update CFG with detected dimensions
            CFG["esm_dim"] = len(sample['esm_embedding'])
            CFG["gvp_dim"] = len(sample['gvp_embedding'])
            CFG["chemberta_dim"] = len(sample['chemberta_embedding'])
            CFG["egnn_dim"] = len(sample['egnn_embedding'])
            
            print("--- CFG Updated from Data ---")
            print(f"Protein (ESM/GVP): {CFG['esm_dim']}/{CFG['gvp_dim']}")
            print(f"Drug (Chem/EGNN): {CFG['chemberta_dim']}/{CFG['egnn_dim']}")
            print(f"ADR Vocab Size: {self.adr_manager.vocab_size}")
            print("-----------------------------")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        # 1. Protein & Drug Features (Standardizing to 1D)
        esm = torch.tensor(row['esm_embedding'], dtype=torch.float32)
        chem = torch.tensor(row['chemberta_embedding'], dtype=torch.float32)
        
        gvp_raw = torch.tensor(row['gvp_embedding'], dtype=torch.float32)
        egnn_raw = torch.tensor(row['egnn_embedding'], dtype=torch.float32)

        # Average pooling if multi-dimensional (jagged check)
        gvp = gvp_raw.mean(dim=0) if gvp_raw.dim() > 1 else gvp_raw
        egnn = egnn_raw.mean(dim=0) if egnn_raw.dim() > 1 else egnn_raw
        
        # 2. ADR Processing
        raw_adr_ids = row['adr_ids'] 
        
        # Use your ADRData class to create the binary multi-hot vector
        adr_binary = torch.tensor(self.adr_manager.encode(raw_adr_ids), dtype=torch.float32)
        
        # 3. ADR Target (Centroid calculation)
        active_indices = torch.where(adr_binary == 1)[0]
        
        if self.adr_map and len(active_indices) > 0:
            # We look up prototypes using the INTEGER indices (0 to 4816)
            target_centroids = [
                self.adr_map[idx.item()] 
                for idx in active_indices 
                if idx.item() in self.adr_map
            ]
            
            if target_centroids:
                adr_target = torch.stack(target_centroids).mean(dim=0)
            else:
                adr_target = torch.zeros(CFG["latent_dim"])
        else:
            adr_target = torch.zeros(CFG["latent_dim"])

        return {
            'esm': esm, 
            'gvp': gvp,
            'chem': chem, 
            'egnn': egnn,
            'adr_target': adr_target,
            'adr_binary': adr_binary # Fixed length 4817 vector
        }

In [127]:
@torch.no_grad()
def update_adr_prototypes(model, loader, device, num_adr_classes):
    """
    Re-calculates the centroid for every ADR using efficient matrix operations.
    """
    model.eval()
    
    # Accumulators on GPU
    adr_sums = torch.zeros((num_adr_classes, CFG["latent_dim"]), device=device)
    adr_counts = torch.zeros(num_adr_classes, device=device)

    for batch in loader:
        # 1. Get current latent embeddings
        esm, gvp = batch['esm'].to(device), batch['gvp'].to(device)
        chem, egnn = batch['chem'].to(device), batch['egnn'].to(device)
        adr_binary = batch['adr_binary'].to(device) # Shape: [Batch, 4817]
        
        z_p, z_d = model(esm, gvp, chem, egnn)
        
        # 2. Define the context vector (Drug + Protein average)
        context_vecs = (z_d + z_p) / 2
        
        # 3. Efficient Accumulation via Matrix Multiplication
        adr_sums += torch.matmul(adr_binary.t(), context_vecs)
        
        # Count occurrences of each ADR in this batch
        adr_counts += adr_binary.sum(dim=0)

    # 4. Average and Normalize
    mask = adr_counts > 0
    new_prototypes = torch.zeros_like(adr_sums)
    
    new_prototypes[mask] = adr_sums[mask] / adr_counts[mask].unsqueeze(1)
    
    # For ADRs never seen, keep them as random/original or keep zero
    # Finally, normalize to the hypersphere
    new_prototypes = F.normalize(new_prototypes, p=2, dim=-1)
    
    return new_prototypes

In [128]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    epoch_dti_loss = 0
    epoch_adr_loss = 0
    
    # Optional: You can adjust these weights if one loss is much larger than the other
    w_dti = 1.0
    w_adr = 100.0  # Prototyping losses (MSE) are often much smaller than Contrastive
    
    for batch in loader:
        # 1. Move all data to GPU
        esm, gvp = batch['esm'].to(device), batch['gvp'].to(device)
        chem, egnn = batch['chem'].to(device), batch['egnn'].to(device)
        adr_target = batch['adr_target'].to(device) 
        
        optimizer.zero_grad()
        
        # 2. --- FORWARD PASS ---
        # z_p: protein latent [batch, 256], z_d: drug latent [batch, 256]
        z_p, z_d = model(esm, gvp, chem, egnn)
        
        # 3. --- LOSS CALCULATION ---
        # Task A: DTI Contrastive (alignment of Drug-Protein pairs)
        loss_dti = criterion.dti_contrastive_loss(z_p, z_d)
        
        # Task B: ADR Prototyping (alignment of Drug to its ADR Centroid)
        loss_adr = criterion.adr_centroid_loss(z_d, adr_target)
        
        # Weighted Total Loss
        total_loss = (w_dti * loss_dti) + (w_adr * loss_adr)
        
        # 4. --- BACKWARD PASS ---
        total_loss.backward()
        
        # Gradient Clipping: Prevents exploding gradients in ESM/GNN backbones
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        # Log the raw (unweighted) losses for monitoring
        epoch_dti_loss += loss_dti.item()
        epoch_adr_loss += loss_adr.item()
        
    avg_dti = epoch_dti_loss / len(loader)
    avg_adr = epoch_adr_loss / len(loader)
    
    return avg_dti, avg_adr

In [129]:
from sklearn.metrics import roc_auc_score, accuracy_score
import numpy as np

@torch.no_grad()
def validate_epoch(model, loader, criterion, device):
    model.eval()
    val_dti_loss = 0
    val_adr_loss = 0
    
    all_scores = []
    all_targets = []
    
    for batch in loader:
        esm, gvp = batch['esm'].to(device), batch['gvp'].to(device)
        chem, egnn = batch['chem'].to(device), batch['egnn'].to(device)
        adr_target = batch['adr_target'].to(device)
        
        z_p, z_d = model(esm, gvp, chem, egnn)
        
        # 1. Losses
        loss_dti = criterion.dti_contrastive_loss(z_p, z_d)
        loss_adr = criterion.adr_centroid_loss(z_d, adr_target)
        val_dti_loss += loss_dti.item()
        val_adr_loss += loss_adr.item()

        # 2. DTI Metrics (Similarity-based)
        z_p_norm = F.normalize(z_p, p=2, dim=-1)
        z_d_norm = F.normalize(z_d, p=2, dim=-1)
        
        # Calculate similarity matrix [Batch x Batch]
        sim_matrix = torch.matmul(z_d_norm, z_p_norm.t()) 
        
        # Positive pairs are the diagonal (Drug i with Protein i)
        pos_scores = torch.diag(sim_matrix).cpu().numpy()
        # Negative pairs are everything else in the batch
        mask = ~torch.eye(sim_matrix.size(0), dtype=torch.bool)
        neg_scores = sim_matrix[mask].cpu().numpy()
        
        all_scores.extend(pos_scores)
        all_targets.extend(np.ones(len(pos_scores)))
        
        all_scores.extend(neg_scores)
        all_targets.extend(np.zeros(len(neg_scores)))

    # 3. Calculate AUC
    auc_score = roc_auc_score(all_targets, all_scores)
    
    # 4. Calculate Binary Accuracy (using 0.5 as a standard threshold for similarity)
    # Note: In contrastive learning, accuracy is often measured as "Is the correct 
    # protein the TOP match for the drug in this batch?"
    preds = np.array(all_scores) > 0.5 
    acc_score = accuracy_score(all_targets, preds)
    
    avg_dti_loss = val_dti_loss / len(loader)
    avg_adr_loss = val_adr_loss / len(loader)
    
    return avg_dti_loss, avg_adr_loss, acc_score, auc_score

In [130]:
# 1. Configuration & Initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_adrs = 4817 

# Load ADR Metadata
adrdf = pd.read_parquet("../../Data/final_rxnorm_meddra_v2.parquet")
id_name_dict = dict(zip(adrdf['meddra_id'], adrdf['meddra_name']))
adr_manager = ADRData(id_name_dict)

# 2. Initial Dataset Setup (LOAD ONCE)
datapath = "train_dti.parquet"
train_dataset = DTIDataset(parquet_path=datapath, adr_manager=adr_manager)

# 3. Model & Initial Prototypes
# (BioMultitaskModel uses dimensions updated by DTIDataset above)
model = BioMultitaskModel(CFG).to(device)
criterion = MultiTaskLoss().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

all_adr_prototypes = torch.randn(num_adrs, CFG["latent_dim"]).to(device)
all_adr_prototypes = F.normalize(all_adr_prototypes, p=2, dim=-1)



val_datapath = "test_dti.parquet" # Or val_dti.parquet
val_dataset = DTIDataset(parquet_path=val_datapath, adr_manager=adr_manager)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)



# 4. The Grand Loop
for epoch in range(1, 51):
    # Step A: Update the map in the EXISTING dataset object
    # This prevents re-reading the parquet from disk every epoch
    train_dataset.adr_map = {i: all_adr_prototypes[i] for i in range(num_adrs)}
    
    # Re-wrap in DataLoader (shuffles the data for the new epoch)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

    # Step B: Train Weights
    dti_loss, adr_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    
    # Step C: Dynamic Prototype Update
    # This recalculates ADR centroids based on the new model weights
    all_adr_prototypes = update_adr_prototypes(model, train_loader, device, num_adrs)
    
    # Step D: Logging
    print(f"--- Epoch {epoch} Complete ---")
    print(f"DTI Contrastive Loss: {dti_loss:.4f} | ADR Prototyping Loss: {adr_loss:.4f}")

    # Step E: Periodic Evaluation
    if epoch % 5 == 0:
        print(">> Running Validation Metrics...")

        val_dataset.adr_map = {i: all_adr_prototypes[i] for i in range(num_adrs)}
        v_dti_loss, v_adr_loss, dti_acc, dti_auc = validate_epoch(model, val_loader, criterion, device)

        print(f"Validation DTI Contrastive Loss: {v_dti_loss:.4f} | Validation ADR Prototyping Loss: {v_adr_loss:.4f}")
        print(f"Validation DTI Accuracy: {dti_acc:.4f} | Validation DTI AUC: {dti_auc:.4f}")
        # Note: You should have a separate val_dataset/val_loader that doesn't shuffle

--- CFG Updated from Data ---
Protein (ESM/GVP): 1280/1024
Drug (Chem/EGNN): 384/256
ADR Vocab Size: 4817
-----------------------------
--- CFG Updated from Data ---
Protein (ESM/GVP): 1280/1024
Drug (Chem/EGNN): 384/256
ADR Vocab Size: 4817
-----------------------------
--- Epoch 1 Complete ---
DTI Contrastive Loss: 3.8169 | ADR Prototyping Loss: 0.0039
--- Epoch 2 Complete ---
DTI Contrastive Loss: 3.5782 | ADR Prototyping Loss: 0.0010
--- Epoch 3 Complete ---
DTI Contrastive Loss: 3.4864 | ADR Prototyping Loss: 0.0006
--- Epoch 4 Complete ---
DTI Contrastive Loss: 3.4461 | ADR Prototyping Loss: 0.0005
--- Epoch 5 Complete ---
DTI Contrastive Loss: 3.4177 | ADR Prototyping Loss: 0.0006
>> Running Validation Metrics...
Validation DTI Contrastive Loss: 4.2404 | Validation ADR Prototyping Loss: 0.0015
Validation DTI Accuracy: 0.9642 | Validation DTI AUC: 0.5192
--- Epoch 6 Complete ---
DTI Contrastive Loss: 3.3861 | ADR Prototyping Loss: 0.0006
--- Epoch 7 Complete ---
DTI Contrastive L

KeyboardInterrupt: 

In [140]:
val_dataset.adr_map = {i: all_adr_prototypes[i] for i in range(num_adrs)}
v_dti_loss, v_adr_loss, dti_acc, dti_auc = validate_epoch(model, val_loader, criterion, device)

print(f"Validation DTI Contrastive Loss: {v_dti_loss:.4f} | Validation ADR Prototyping Loss: {v_adr_loss:.4f}")
print(f"Validation DTI Accuracy: {dti_acc:.4f} | Validation DTI AUC: {dti_auc:.4f}")

Validation DTI Contrastive Loss: 3.4563 | Validation ADR Prototyping Loss: 0.0004
Validation DTI Accuracy: 0.9843 | Validation DTI AUC: 0.7835


In [137]:
@torch.no_grad()
def predict_by_ids(model, dataset, adr_manager, all_prototypes, protein_id, drug_id, device, threshold=0.75):
    """
    Takes Protein and Drug IDs, finds them in the dataset, 
    and returns DTI prediction + ADR profile.
    """
    model.eval()
    
    # 1. Find the index for the IDs in your dataset
    # This assumes your dataset has a way to map IDs to row indices
    try:
        p_idx = dataset.data[dataset.data['target_uniprot_id'] == protein_id].index[0]
        d_idx = dataset.data[dataset.data['rxcui'] == drug_id].index[0]
    except IndexError:
        return "ID not found in the provided dataset."

    # 2. Get the processed embeddings from the dataset
    # We use the dataset's __getitem__ logic to ensure processing is identical
    p_data = dataset[p_idx]
    d_data = dataset[d_idx]

    # 3. Prepare Tensors
    esm = p_data['esm'].unsqueeze(0).to(device)
    gvp = p_data['gvp'].unsqueeze(0).to(device)
    chem = d_data['chem'].unsqueeze(0).to(device)
    egnn = d_data['egnn'].unsqueeze(0).to(device)

    # 4. Forward Pass -> Latent Space
    z_p, z_d = model(esm, gvp, chem, egnn)
    
    # Normalize for Cosine Similarity
    z_p = F.normalize(z_p, p=2, dim=-1)
    z_d = F.normalize(z_d, p=2, dim=-1)

    # 5. DTI Prediction (Similarity)
    dti_score = torch.sum(z_p * z_d, dim=-1).item()
    interacts = dti_score > threshold

    # 6. ADR Prediction (Similarity to all prototypes)
    # prototypes shape: [4817, 256]
    adr_sims = torch.matmul(z_d, all_prototypes.t())
    scores, indices = torch.topk(adr_sims, k=20)

    # Map indices to names
    top_adrs = []
    for idx in indices[0]:
        name = adr_manager.id_to_name.get(adr_manager.idx_to_id[idx.item()], "Unknown")
        top_adrs.append(name)

    return {
        "DTI_Similarity": round(dti_score, 4),
        "Interacts": interacts,
        "Top_5_Predicted_ADRs": top_adrs
    }

In [139]:
# Example Usage:
result = predict_by_ids(
    model=model, 
    dataset=train_dataset, 
    adr_manager=adr_manager, 
    all_prototypes=all_adr_prototypes, 
    protein_id="Q8NI60", 
    drug_id="1603296", 
    device=device
)

print(f"Prediction: {result['Interacts']} (Score: {result['DTI_Similarity']})")
print(f"Likely Side Effects: {', '.join(result['Top_5_Predicted_ADRs'])}")

Prediction: False (Score: 0.4309)
Likely Side Effects: Sputum bloody, Opportunistic infection, Oral candidiasis, Pain in limb, Hemiplegia, Eye complication associated with device, Keratoconus, Corneal striae, Meibomian gland dysfunction, Corneal epithelium defect, Lymphadenopathy, Thrombocytopenic purpura, Ear disorder, Quadriplegia, Sore throat, Skin ulcer, Sinus pause, Haemolytic anaemia, Nephrotoxicity, Language disorder


In [86]:
def validate_dataset_dimensions(parquet_path):
    import pandas as pd
    from collections import Counter
    
    print(f"--- Loading {parquet_path} for validation ---")
    df = pd.read_parquet(parquet_path)
    
    columns_to_check = ['esm_embedding', 'gvp_embedding', 'chemberta_embedding', 'egnn_embedding']
    all_clear = True

    for col in columns_to_check:
        if col not in df.columns:
            print(f"❌ Column '{col}' missing from Parquet file!")
            continue
            
        # Get the length of every list/array in this column
        lengths = df[col].apply(lambda x: len(x) if x is not None else 0)
        counts = Counter(lengths)
        
        if len(counts) > 1:
            all_clear = False
            print(f"⚠️  Dimension Mismatch in '{col}':")
            for length, count in counts.items():
                print(f"   - {count} rows have length {length}")
            
            # Find the index of the first "outlier"
            common_len = counts.most_common(1)[0][0]
            outlier_idx = lengths[lengths != common_len].index[0]
            print(f"   - Example outlier at index: {outlier_idx}")
        else:
            print(f"✅ '{col}' is consistent (Length: {list(counts.keys())[0]})")

    if all_clear:
        print("\n✨ All embedding columns have consistent dimensions. The RuntimeError likely came from the DataLoader trying to stack 'true_adr_indices'.")
    else:
        print("\n🚨 Please fix the mismatched rows or use the 'fix_dim' padding/truncating method in the Dataset class.")

# Run the test
validate_dataset_dimensions("../../Data/processed_dti_dataset.parquet")

--- Loading ../../Data/processed_dti_dataset.parquet for validation ---
✅ 'esm_embedding' is consistent (Length: 1280)
✅ 'gvp_embedding' is consistent (Length: 1024)
✅ 'chemberta_embedding' is consistent (Length: 384)
✅ 'egnn_embedding' is consistent (Length: 256)

✨ All embedding columns have consistent dimensions. The RuntimeError likely came from the DataLoader trying to stack 'true_adr_indices'.


In [ ]:


# 1. Initialize Model, Optimizer, and Loss
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BioMultitaskModel(CFG).to(device)
criterion = MultiTaskLoss(temperature=0.07).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

# 2. Training Function for One Epoch
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    epoch_dti_loss = 0
    epoch_adr_loss = 0
    
    for batch in loader:
        # Move all data to GPU
        esm, gvp = batch['esm'].to(device), batch['gvp'].to(device)
        chem, egnn = batch['chem'].to(device), batch['egnn'].to(device)
        adr_target = batch['adr_target'].to(device) # The "Middle Point" target
        
        optimizer.zero_grad()
        
        # --- FORWARD PASS ---
        # Get fused protein and drug embeddings in the shared 256-dim space
        z_p, z_d = model(esm, gvp, chem, egnn)
        
        # --- LOSS CALCULATION ---
        # Task A: DTI Contrastive (pulls drug and its target protein together)
        loss_dti = criterion.dti_contrastive_loss(z_p, z_d)
        
        # Task B: ADR Prototyping (pulls drug toward its ADR "middle point")
        loss_adr = criterion.adr_centroid_loss(z_d, adr_target)
        
        # Total Loss (you can weight these, e.g., 0.7 * dti + 0.3 * adr)
        total_loss = loss_dti + loss_adr
        
        # --- BACKWARD PASS ---
        total_loss.backward()
        optimizer.step()
        
        epoch_dti_loss += loss_dti.item()
        epoch_adr_loss += loss_adr.item()
        
    return epoch_dti_loss / len(loader), epoch_adr_loss / len(loader)

# 3. Execution Loop
for epoch in range(50):
    dti_l, adr_l = train_epoch(model, train_loader, optimizer, criterion, device)
    print(f"Epoch {epoch}: DTI Loss: {dti_l:.4f} | ADR Loss: {adr_l:.4f}")

In [99]:
from sklearn.model_selection import train_test_split

# Load your full processed data
full_df = pd.read_parquet("../../Data/processed_dti_dataset.parquet")

# Split: 80% Train, 20% Test (or 80/10/10 for Train/Val/Test)
train_df, test_df = train_test_split(full_df, test_size=0.2, random_state=42)

# Save them so your loaders can find them
train_df.to_parquet("train_dti.parquet")
test_df.to_parquet("test_dti.parquet")